# StyleGAN2: Brain Tumor MRI Synthesis (512x512)

This notebook trains a StyleGAN2 model to generate realistic brain MRI images with tumors. 

### Key Features:
- **High Resolution**: 512x512 output.
- **Artifact-Free**: Uses weight demodulation and R1 regularization to eliminate "circle" biases.
- **Class-Wise Support**: Can be trained on specific tumor types (glioma, meningioma, pituitary).
- **Kaggle Optimized**: Memory-efficient settings for 15GB RAM.
- **Automated Inference**: Generates a synthetic dataset and zips it for download.

In [2]:
!pip install -q stylegan2_pytorch torchinfo
!pip install matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 864.4 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 35.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 54.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 30.0 MB/s eta 0:00:00


In [25]:
import os
import shutil

# 🔹 Set your source folders (top-level folders)
source_dirs = [
    "/home/rehan/DeepFakes/Datasets/briscdataset/brisc2025",
    "/home/rehan/DeepFakes/Datasets/masoudnickparvar/brain-tumor-mri-dataset",
    "/home/rehan/DeepFakes/Datasets/unidpro/brain-cancer-dataset",
    "/home/rehan/DeepFakes/Datasets/brain-tumor-mri-dataset"
]

# 🔹 Destination folder
destination_dir = "/home/rehan/DeepFakes/Datasets/CombinedDataset"

# Create destination folder if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Supported image extensions
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

counter = 1  # for renaming files to avoid conflicts

for src_dir in source_dirs:
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.lower().endswith(image_extensions):
                src_path = os.path.join(root, file)

                # Create unique filename
                ext = os.path.splitext(file)[1]
                new_name = f"image_{counter}{ext}"
                dest_path = os.path.join(destination_dir, new_name)

                # Copy file
                shutil.copy2(src_path, dest_path)

                # print(f"Copied: {src_path} -> {dest_path}")
                counter += 1
    print(counter - 1, "images copied from", src_dir)
print("✅ All images combined successfully!")

6000 images copied from /home/rehan/DeepFakes/Datasets/briscdataset/brisc2025
13200 images copied from /home/rehan/DeepFakes/Datasets/masoudnickparvar/brain-tumor-mri-dataset
13550 images copied from /home/rehan/DeepFakes/Datasets/unidpro/brain-cancer-dataset
18550 images copied from /home/rehan/DeepFakes/Datasets/brain-tumor-mri-dataset
✅ All images combined successfully!


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("briscdataset/brisc2025")

print("Path to dataset files:", path)


/home/rehan/DeepFakes/gpu_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 250M/250M [00:13<00:00, 18.8MB/s] 

Extracting files...


Path to dataset files: /home/rehan/.cache/kagglehub/datasets/briscdataset/brisc2025/versions/6


In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("unidpro/brain-cancer-dataset")

print("Path to dataset files:", path)

100%|██████████| 119M/119M [00:07<00:00, 15.7MB/s] 

Extracting files...


Path to dataset files: /home/rehan/.cache/kagglehub/datasets/unidpro/brain-cancer-dataset/versions/1


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

100%|██████████| 157M/157M [00:09<00:00, 17.5MB/s] 

Extracting files...


Path to dataset files: /home/rehan/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


In [10]:
import os
import shutil
import random
from functools import partial
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn as nn
import torch
import stylegan2_pytorch.stylegan2_pytorch as s2p
from stylegan2_pytorch import Trainer, ModelLoader
from torchvision.utils import save_image, make_grid

# Check for CUDA (GPU) or use CPU if no GPU is available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [11]:
# Grayscale runtime patch: keep the dataset and GAN in 1-channel mode.
def convert_image_to_grayscale(image):
    return image.convert('L') if image.mode != 'L' else image

class keep_single_channel(object):
    def __init__(self, transparent):
        self.transparent = transparent

    def __call__(self, tensor):
        channels = tensor.shape[0]
        if channels == 1:
            return tensor
        if channels >= 2:
            return tensor[:1]
        raise Exception(f'image with invalid number of channels given {channels}')

def grayscale_dataset_init(self, folder, image_size, transparent=False, aug_prob=0.):
    super(s2p.Dataset, self).__init__()
    self.folder = folder
    self.image_size = image_size
    self.paths = [p for ext in s2p.EXTS for p in Path(f'{folder}').glob(f'**/*.{ext}')]
    assert len(self.paths) > 0, f'No images were found in {folder} for training'

    self.transform = s2p.transforms.Compose([
        s2p.transforms.Lambda(convert_image_to_grayscale),
        s2p.transforms.Lambda(partial(s2p.resize_to_minimum_size, image_size)),
        s2p.transforms.Resize(image_size),
        s2p.RandomApply(aug_prob, s2p.transforms.RandomResizedCrop(image_size, scale=(0.5, 1.0), ratio=(0.98, 1.02)), s2p.transforms.CenterCrop(image_size)),
        s2p.transforms.ToTensor(),
        s2p.transforms.Lambda(keep_single_channel(transparent))
    ])

def grayscale_rgbblock_init(self, latent_dim, input_channel, upsample, rgba=False):
    nn.Module.__init__(self)
    self.input_channel = input_channel
    self.to_style = nn.Linear(latent_dim, input_channel)
    self.conv = s2p.Conv2DMod(input_channel, 1, 1, demod=False)
    self.upsample = nn.Sequential(
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
        s2p.Blur()
    ) if upsample else None

def grayscale_discriminator_init(self, image_size, network_capacity=16, fq_layers=[], fq_dict_size=256, attn_layers=[], transparent=False, fmap_max=512):
    nn.Module.__init__(self)
    num_layers = int(s2p.log2(image_size) - 1)
    num_init_filters = 1

    filters = [num_init_filters] + [(network_capacity * 4) * (2 ** i) for i in range(num_layers + 1)]
    set_fmap_max = partial(min, fmap_max)
    filters = list(map(set_fmap_max, filters))
    chan_in_out = list(zip(filters[:-1], filters[1:]))

    blocks = []
    attn_blocks = []
    quantize_blocks = []

    for ind, (in_chan, out_chan) in enumerate(chan_in_out):
        num_layer = ind + 1
        is_not_last = ind != (len(chan_in_out) - 1)
        block = s2p.DiscriminatorBlock(in_chan, out_chan, downsample=is_not_last)
        blocks.append(block)
        attn_fn = s2p.attn_and_ff(out_chan) if num_layer in attn_layers else None
        attn_blocks.append(attn_fn)
        quantize_fn = s2p.PermuteToFrom(s2p.VectorQuantize(out_chan, fq_dict_size)) if num_layer in fq_layers else None
        quantize_blocks.append(quantize_fn)

    self.blocks = nn.ModuleList(blocks)
    self.attn_blocks = nn.ModuleList(attn_blocks)
    self.quantize_blocks = nn.ModuleList(quantize_blocks)
    chan_last = filters[-1]
    latent_dim = 2 * 2 * chan_last
    self.final_conv = nn.Conv2d(chan_last, chan_last, 3, padding=1)
    self.flatten = s2p.Flatten()
    self.to_logit = nn.Linear(latent_dim, 1)

s2p.convert_transparent_to_rgb = convert_image_to_grayscale
s2p.expand_greyscale.__call__ = keep_single_channel.__call__
s2p.Dataset.__init__ = grayscale_dataset_init
s2p.RGBBlock.__init__ = grayscale_rgbblock_init
s2p.Discriminator.__init__ = grayscale_discriminator_init
print('Grayscale patch applied: dataset loads L images and GAN outputs 1 channel.')

Grayscale patch applied: dataset loads L images and GAN outputs 1 channel.


In [3]:
sample_path = TEMP_DATA_DIR / 'img_00000.png'
with Image.open(sample_path) as img:
    gray = img.convert('L')
    tensor = torch.from_numpy(np.array(gray)).float().unsqueeze(0) / 255.0
    batch_tensor = tensor.unsqueeze(0)

    print(f'Sample path: {sample_path}')
    print(f'PIL mode: {gray.mode}')
    print(f'NumPy shape: {np.array(gray).shape}')
    print(f'Tensor shape (C, H, W): {tuple(tensor.shape)}')
    print(f'Batch tensor shape (N, C, H, W): {tuple(batch_tensor.shape)}')
    plt.imshow(gray, cmap='gray')
    plt.title('Tensor Shape Sanity Check')
    plt.axis('off')
    plt.show()

NameError: name 'TEMP_DATA_DIR' is not defined

## 1. Configuration
Adjust these parameters based on your dataset and available time.

In [12]:
# Dataset Paths
RUN_NAME = 'mri_tumor_gan_gray'
DATA_ROOT = Path('../Datasets/CombinedDataset')
WORK_DIR = Path('../StyleGAN')
MODELS_DIR = WORK_DIR / 'models'
CHECKPOINTS_DIR = WORK_DIR / 'checkpoints'
RESULTS_DIR = WORK_DIR / 'results'
GEN_OUT_DIR = Path('../StyleGAN/Outputs')
TEMP_DATA_DIR = Path('../Datasets/CleanCombinedDatasetGray')
TEMP_DATA_DIR.mkdir(exist_ok=True)

for d in [WORK_DIR, MODELS_DIR, CHECKPOINTS_DIR, RESULTS_DIR, GEN_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# High Res Training Params
IMAGE_SIZE = 512
BATCH_SIZE = 2 # Small batch size for 512x512 on 15GB RAM
GRAD_ACCUM = 4 # Total batch size = BATCH_SIZE * GRAD_ACCUM = 8
NETWORK_CAPACITY = 16
STEPS = 10000 # Increase for better quality (e.g., 50000+)
SAVE_EVERY = 100

print(f"Resolution: {IMAGE_SIZE}x{IMAGE_SIZE} | Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM}")

Resolution: 512x512 | Effective Batch Size: 8


## 2. Data Preparation
StyleGAN2 requires a folder of images. We will scan your `mri-dataset` and create a clean flat directory for training.

In [13]:


def count_images(root):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    return sum(1 for p in Path(root).rglob('*') if p.suffix.lower() in exts)

# def prepare_dataset(src_root, dst_root):
#     exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
#     src_root = Path(src_root)
#     dst_root = Path(dst_root)

#     if not src_root.exists():
#         raise FileNotFoundError(f'Dataset path does not exist: {src_root}')

#     image_paths = [p for p in src_root.rglob('*') if p.suffix.lower() in exts]
#     print(f'Found {len(image_paths)} source images in {src_root}')

#     count = 0
#     failed = 0
#     for p in image_paths[:]:
#         try:
#             with Image.open(p) as img:
#                 img = img.convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
#                 img.save(dst_root / f'img_{count:05d}.png')
#                 if count == 0:
#                     plt.imshow(img, cmap='gray')
#                     plt.title('Sample Preprocessed Grayscale Image')
#                     plt.axis('off')
#                     plt.show()
#                 count += 1
#         except Exception:
#             failed += 1

#     print(f'Successfully prepared {count} images at {IMAGE_SIZE}x{IMAGE_SIZE}. Failed reads: {failed}')
#     return count

# img_count = prepare_dataset(DATA_ROOT, TEMP_DATA_DIR)
# if img_count == 0:
#     print('Warning: 0 processed images. Training cell will fallback to DATA_ROOT if it contains images.')

In [18]:
# intall the cv2
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 64.1 MB/s eta 0:00:00:00:0100:01


## 3. Training StyleGAN2
This uses consistent normalization and weight demodulation to ensure no artifacts.

In [15]:
# 3A. Checkpoint Settings and Utilities
CKPT_EVERY = SAVE_EVERY  # Save checkpoint every N steps

def save_checkpoint_safe(trainer, global_step):
    if not hasattr(trainer, 'save'):
        return False
    # stylegan2_pytorch expects checkpoint index, not raw global step.
    ckpt_index = max(1, int(global_step // CKPT_EVERY))
    try:
        trainer.save(ckpt_index)
        return True
    except TypeError:
        # Some versions use save() without arguments
        trainer.save()
        return True
    except Exception as e:
        print(f'Checkpoint save skipped at step {global_step}: {e}')
        return False

def list_checkpoint_files(models_dir, run_name=RUN_NAME):
    run_dir = Path(models_dir) / run_name
    if not run_dir.exists():
        return []
    files = sorted(run_dir.glob('model_*.pt'))
    return files

def infer_resume_step(models_dir, run_name=RUN_NAME, save_every=100):
    ckpts = list_checkpoint_files(models_dir, run_name=run_name)
    if not ckpts:
        return 0
    latest_num = int(ckpts[-1].stem.split('_')[-1])
    # Handle both naming styles safely:
    # - model_21.pt (checkpoint index) -> 2100 steps
    # - model_2100.pt (raw steps) -> 2100 steps
    if latest_num % save_every == 0 and latest_num >= save_every:
        return latest_num
    return latest_num * save_every

print(f'Checkpoint interval: every {CKPT_EVERY} steps')
existing_ckpts = list_checkpoint_files(CHECKPOINTS_DIR, run_name=RUN_NAME)
for p in existing_ckpts[:]:
    print(p)

print(f'Existing checkpoints found: {len(existing_ckpts)}')
if existing_ckpts:
    print(f'Latest checkpoint: {existing_ckpts[-6]}')
    print(f'Inferred resume step: {infer_resume_step(CHECKPOINTS_DIR, save_every=CKPT_EVERY)}')

Checkpoint interval: every 100 steps
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_0.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_1.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_10.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_11.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_12.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_13.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_14.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_15.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_16.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_17.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_18.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_19.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_2.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_20.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_21.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_22.pt
../StyleGAN/checkpoints/mri_tumor_gan_gray/model_23.pt
../StyleGAN/checkpoints/mri_tum

In [20]:
import inspect

trainer_kwargs = dict(
    name=RUN_NAME,
    results_dir=str(RESULTS_DIR),
    models_dir=str(CHECKPOINTS_DIR),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    gradient_accumulate_every=GRAD_ACCUM,
    network_capacity=NETWORK_CAPACITY,
    lr=2e-4,
    save_every=SAVE_EVERY,
    data=str(TEMP_DATA_DIR),
    fp16=False  # Apex is unavailable in most Kaggle runtimes
)

# Keep only kwargs supported by the installed stylegan2_pytorch version
trainer_init_params = set(inspect.signature(Trainer.__init__).parameters.keys())
if 'num_train_steps' in trainer_init_params:
    trainer_kwargs['num_train_steps'] = STEPS
if 'save_every' not in trainer_init_params:
    trainer_kwargs.pop('save_every', None)

trainer = Trainer(**{k: v for k, v in trainer_kwargs.items() if k in trainer_init_params})

# Choose processed folder if available, else fallback to raw dataset root
train_src = TEMP_DATA_DIR if count_images(TEMP_DATA_DIR) > 0 else DATA_ROOT
num_train_images = count_images(train_src)
print(f'Using training source: {train_src} | images found: {num_train_images}')

if num_train_images == 0:
    raise RuntimeError(
        f'No images found in {train_src}. Update DATA_ROOT to the correct dataset folder.'
    )

trainer.set_data_src(str(train_src))

# Always try to resume from latest checkpoint automatically
resume_step = infer_resume_step(CHECKPOINTS_DIR, run_name=RUN_NAME, save_every=CKPT_EVERY)
print(f'Attempting to resume from checkpoint at global step {resume_step}...')
if hasattr(trainer, 'load'):
    ckpts = list_checkpoint_files(CHECKPOINTS_DIR, run_name=RUN_NAME)
    checkpoint_loaded = False
    
    # Try loading checkpoints in reverse order (newest to oldest)
    for ckpt_path in reversed(ckpts):
        ckpt_index = int(ckpt_path.stem.split('_')[-1])
        try:
            print(f'Trying to load checkpoint {ckpt_path.name}...')
            trainer.load(ckpt_index)
            # Normalize tracked step so logs and loops start at the correct global step.
            resume_step = infer_resume_step(CHECKPOINTS_DIR, run_name=RUN_NAME, save_every=CKPT_EVERY)
            trainer.steps = resume_step
            print(f'✓ Successfully resumed from checkpoint {ckpt_path.name} at global step {resume_step}.')
            checkpoint_loaded = True
            break
        except Exception as e:
            print(f'✗ Failed to load checkpoint {ckpt_path.name}: {e}')
            print(f'  Trying previous checkpoint...')
    
    if not checkpoint_loaded:
        print('No usable checkpoint found. Starting fresh.')
        resume_step = 0

# Newer versions train for num_train_steps internally; older versions need manual looping
if 'num_train_steps' in trainer_init_params and 'save_every' in trainer_init_params:
    if resume_step > 0:
        print('Using built-in train(); progress logs are handled by stylegan2_pytorch.')
    trainer.train()
else:
    target_steps = STEPS
    if resume_step >= target_steps:
        print(f'Model already reached target steps ({resume_step} >= {target_steps}).')
    else:
        print(f'Running manual training loop from step {resume_step + 1} to {target_steps}...')
        for global_step in range(resume_step + 1, target_steps + 1):
            trainer.train()

            # Force periodic checkpoints in compatibility/manual mode
            if global_step % CKPT_EVERY == 0:
                if save_checkpoint_safe(trainer, global_step):
                    print(f'Checkpoint saved at step {global_step}')

            if global_step % 100 == 0 or global_step == resume_step + 1:
                print(f'Step {global_step}/{target_steps}')

        # Save final checkpoint snapshot at end of run
        if save_checkpoint_safe(trainer, target_steps):
            print('Final checkpoint saved.')

Using training source: ../Datasets/CleanCombinedDatasetGray | images found: 18551
Attempting to resume from checkpoint at global step 4000...
Trying to load checkpoint model_40.pt...
loading from version 1.9.0
✓ Successfully resumed from checkpoint model_40.pt at global step 4000.
Running manual training loop from step 4001 to 10000...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 9.77 GiB of which 2.25 MiB is free. Process 5705 has 3.42 GiB memory in use. Including non-PyTorch memory, this process has 6.34 GiB memory in use. Of the allocated memory 5.96 GiB is allocated by PyTorch, and 105.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 4. Inference & Generation
Generate 512x512 realistic tumor images using the trained weights.

In [14]:
from stylegan2_pytorch.stylegan2_pytorch import image_noise

@torch.no_grad()
def generate_dataset(n=100, trunc_psi=0.5):
    # Use Trainer directly so inference reads the same checkpoint path used in training.
    infer_trainer = Trainer(
        name=RUN_NAME,
        models_dir=str(CHECKPOINTS_DIR),
        image_size=IMAGE_SIZE,
        network_capacity=NETWORK_CAPACITY,
        batch_size=1,
        fp16=False
    )
    infer_trainer.load(-1)
    infer_trainer.GAN.eval()

    print(f'Loaded model at image_size={infer_trainer.image_size}, steps={infer_trainer.steps}')
    print(f"Generating {n} synthetic MRI scans...")

    latent_dim = infer_trainer.GAN.G.latent_dim
    num_layers = infer_trainer.GAN.G.num_layers

    for i in range(n):
        z = torch.randn(1, latent_dim, device=DEVICE)
        style = [(z, num_layers)]
        n_noise = image_noise(1, infer_trainer.image_size, device=0 if DEVICE.type == 'cuda' else 'cpu')

        images = infer_trainer.generate_truncated(
            infer_trainer.GAN.SE,
            infer_trainer.GAN.GE,
            style,
            n_noise,
            trunc_psi=trunc_psi,
            num_image_tiles=1
        )

        if images.shape[-2:] != (IMAGE_SIZE, IMAGE_SIZE):
            print(f"Warning: Generated image dimensions are {images.shape[-2:]}, expected ({IMAGE_SIZE}, {IMAGE_SIZE}).")

        save_image(images, GEN_OUT_DIR / f"synthetic_tumor_{i:04d}.png")

    print('Done.', GEN_OUT_DIR)

generate_dataset(n=10, trunc_psi=0.5)

continuing from previous epoch - 40
loading from version 1.9.0
unable to load save model. please try downgrading the package to the version specified by the saved model


RuntimeError: Error(s) in loading state_dict for StyleGAN2:
	size mismatch for G.blocks.0.to_rgb.conv.weight: copying a param with shape torch.Size([1, 512, 1, 1]) from checkpoint, the shape in current model is torch.Size([3, 512, 1, 1]).
	size mismatch for G.blocks.1.to_rgb.conv.weight: copying a param with shape torch.Size([1, 512, 1, 1]) from checkpoint, the shape in current model is torch.Size([3, 512, 1, 1]).
	size mismatch for G.blocks.2.to_rgb.conv.weight: copying a param with shape torch.Size([1, 512, 1, 1]) from checkpoint, the shape in current model is torch.Size([3, 512, 1, 1]).
	size mismatch for G.blocks.3.to_rgb.conv.weight: copying a param with shape torch.Size([1, 512, 1, 1]) from checkpoint, the shape in current model is torch.Size([3, 512, 1, 1]).
	size mismatch for G.blocks.4.to_rgb.conv.weight: copying a param with shape torch.Size([1, 256, 1, 1]) from checkpoint, the shape in current model is torch.Size([3, 256, 1, 1]).
	size mismatch for G.blocks.5.to_rgb.conv.wei

## 5. Download Results
Zip the generated images for your classification task.

In [23]:
import shutil
from IPython.display import FileLink

output_zip = '/kaggle/working/StyleGAN_Synthetic_MRIs'
if GEN_OUT_DIR.exists():
    shutil.make_archive(output_zip, 'zip', GEN_OUT_DIR)
    display(FileLink(output_zip + '.zip'))

PermissionError: [Errno 13] Permission denied: '/kaggle'